# ------------------------------------------------
## **PRE-WORKSHOP - BXL CUTTING GARDEN**
### DIVING INTO PRE-PROCESSING & SIGNAL PROCESSING
# ------------------------------------------------

#### IMPORT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mne
import scipy
import os

%matplotlib qt 

#### ON THE MENU

1. Raw EEG object creation
2. EEG visualization
3. Pre-processing:
   - resampling
   - re-referencing
   - filtering
   - artifact rejection
   - ICA
4. Signal analysis:
   - epoching
   - baseline correction
   - ERP
   - time-frequency analysis

##### EXPERIMENTAL PROTOCOL

>>> **Does the brain distinguish faces from landscapes visual categories?** 

![Expe.png](Expe.png)

###### --------------
- 20 min EEG recording
- 21 channels including mastoids
- Sampling rate: 256 Hz


- 2 visual conditions:
  - Face
  - Landscape
- Each trial:
  - fixation: 500 ms
  - stimulus: 500 ms
  - washout: 1500-2000 ms
- Total trials: 400

###### ---------------

### **1. RAW OBJECT MANIP**

#### PARAMETERS

In [ ]:
# Parameters for EEG data simulation
n_channels = 21  # Number of EEG channels
sampling_rate = 256  # Sampling frequency in Hz
duration = 20 * 60  # Recording duration in seconds
n_samples = sampling_rate * duration  # Total number of samples
n_triggers = 400  # Total number of triggers

# Definition of EEG channels
channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'A1', 'A2', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']

n_perm = 250

#### SIMULATION

In [ ]:
def pink_noise(n_samples, n_channels):

    freqs = np.fft.rfftfreq(n_samples, d = 1/sampling_rate)
    amplitudes = 1 / (freqs + 1e-10)
    phases = np.random.uniform(0, 2*np.pi, (n_channels, len(freqs)))
    spectrum = amplitudes * (np.cos(phases) + 1j * np.sin(phases))
    noise = np.fft.irfft(spectrum, n_samples)
    noise = scipy.stats.zscore(noise, axis = 1) * 1e-6

    return noise


def generate_data_set():

    EEG_data = pink_noise(n_samples, n_channels)

    # Add 50 Hz noise
    f_noise = 50
    amplitude_50hz = EEG_data.std()/100

    time = np.arange(n_samples) / sampling_rate
    interference = amplitude_50hz * np.sin(2 * np.pi * f_noise * time)

    EEG_data += interference

    # Add triggers
    trigger_vector = np.zeros(n_samples)

    trigger_times = np.cumsum(np.random.uniform(2.5, 3, n_triggers))  
    trig_idxs = np.round(trigger_times * sampling_rate).astype(int)

    types = np.concatenate([np.ones((int(len(trig_idxs)/2))), np.ones((int(len(trig_idxs)/2)))+1])
    np.random.shuffle(types)
    trigger_vector[trig_idxs] = types
    trig_type = trigger_vector[trigger_vector != 0]

    # Add eye blinks
    blink_amplitude = 1
    blink_duration = 0.5
    blink_samples = int(blink_duration * sampling_rate)
    blink_waveform = np.exp(-np.linspace(-2, 2, blink_samples)**2)
    blink_weights = np.array([1.0 if ch in ['Fp1', 'Fp2', 'F3', 'F4'] else 0.1 for ch in channels])

    blink_times = np.cumsum(np.random.uniform(0, 12, int(n_triggers/2)))
    blink_idxs = np.round(blink_times * sampling_rate).astype(int)

    blink_sig = np.zeros((n_samples))

    for idx in blink_idxs:
        if idx + blink_samples < n_samples:
            blink_sig[idx:idx + blink_samples] = blink_waveform * blink_amplitude

    blink_sig *= EEG_data.std()

    for ch_i, ch in enumerate(channels):
        EEG_data[ch_i,:] += blink_sig * blink_weights[ch_i]

    # Add ERP signals
    p1_amplitude = 0.5
    n1_amplitude = -0.4
    erp_duration = 0.1
    p1_samples = n1_samples = int(erp_duration * sampling_rate)

    p1_waveform = p1_amplitude * np.exp(-np.linspace(-2, 2, p1_samples)**2)
    n1_waveform = n1_amplitude * np.exp(-np.linspace(-2, 2, n1_samples)**2)
        

    erp_weights = np.array([1.0 if ch in ['O1', 'O2'] else 0.7 if ch in ['T5', 'P3', 'Pz', 'P4', 'T6'] else 0 for ch in channels])

    erp_sig = np.zeros((n_samples))

    for idx, t in zip(trig_idxs, types):
        factor = 1 if t == 1 else 0.5
        p1_idx = idx + int(0.1 * sampling_rate)
        n1_idx = idx + int(0.2 * sampling_rate)
        if p1_idx + p1_samples < n_samples:
            erp_sig[p1_idx:p1_idx + p1_samples] = p1_waveform * factor
        if n1_idx + n1_samples < n_samples:
            erp_sig[n1_idx:n1_idx + n1_samples] = n1_waveform * factor

    erp_sig *= EEG_data.std()/5

    for ch_i, ch in enumerate(channels):
        EEG_data[ch_i,:] += erp_sig * erp_weights[ch_i]


    # Add frequency-specific signals (theta and gamma)
    theta_duration = 0.750
    gamma_duration = 0.300
    theta_samples = int(theta_duration * sampling_rate)
    gamma_samples = int(gamma_duration * sampling_rate)
    theta_freq = 6
    gamma_freq = 80

    theta_waveform = np.exp(-np.linspace(-2, 2, theta_samples)**2) * np.sin(2 * np.pi * theta_freq * np.linspace(0, theta_duration, theta_samples))
    gamma_waveform = np.exp(-np.linspace(-2, 2, gamma_samples)**2) * np.sin(2 * np.pi * gamma_freq * np.linspace(0, gamma_duration, gamma_samples))

    jitter = 0.15*sampling_rate

    freq_weights = np.array([1.0 if ch in ['O1', 'O2'] else 0.0 for ch in channels])
    freq_sig = np.zeros((n_samples))

    jitter_list = np.array([np.random.randint(low = trig_i - jitter, high = trig_i + jitter, size = 1)[0] for trig_i in trig_idxs])

    for idx, t in zip(jitter_list, types):
        factor = 1 if t == 1 else 0.5
        gamma_idx = idx + int(0.1 * sampling_rate)
        if gamma_idx + gamma_samples < n_samples:
            freq_sig[gamma_idx:gamma_idx + gamma_samples] = gamma_waveform * factor

    freq_sig *= EEG_data.std()/50

    for ch_i, ch in enumerate(channels):
        EEG_data[ch_i,:] += freq_sig * freq_weights[ch_i]

    freq_sig = np.zeros((n_samples))
    freq_weights = np.array([1.0 if ch in ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'O1', 'O2'] else 0.0 for ch in channels])
    jitter_list = np.array([np.random.randint(low = trig_i - jitter, high = trig_i + jitter, size = 1)[0] for trig_i in trig_idxs])

    for idx, t in zip(jitter_list, types):
        factor = 1 if t == 1 else 0.5
        theta_idx = idx + int(0.2 * sampling_rate)
        if theta_idx + theta_samples < n_samples:
            freq_sig[theta_idx:theta_idx + theta_samples] = theta_waveform * factor

    freq_sig *= EEG_data.std()/10

    for ch_i, ch in enumerate(channels):
        EEG_data[ch_i,:] += freq_sig * freq_weights[ch_i]


    # Add muscle artifacts
    muscle_start = (trig_idxs[-1] + int(10 * sampling_rate)) 
    muscle_duration = 7.5
    muscle_samples = int(muscle_duration * sampling_rate) 

    if muscle_start + muscle_samples < n_samples: # Random high-frequency activity 
        muscle = np.random.randn(muscle_samples) # Band-pass filter: approximately EMG-like 
        b, a = scipy.signal.butter(4, [20, 100], btype = 'bandpass', fs = sampling_rate) 
        muscle = scipy.signal.filtfilt(b, a, muscle) # Smooth beginning and end of artifact 
        envelope = np.hanning(muscle_samples) 
        muscle *= envelope 
        muscle *= EEG_data.std() * 5 # Stronger in frontal chan
        muscle_weights = np.array([1.0 if ch in [ 'Fp1', 'Fp2', 'F3', 'F4'] 
                                   else 0.05 for ch in channels]) 

        for ch_i in range(n_channels): 
            EEG_data[ch_i, muscle_start:muscle_start + muscle_samples] += (muscle * muscle_weights[ch_i])


    trig_idxs = trig_idxs.astype('int')
    trig_type = trig_type.astype('int')

    return EEG_data, trig_idxs, trig_type


def plot_PSD(x):
    hzPxx, Pxx = scipy.signal.welch(x, fs = sampling_rate, window = 'hann', nperseg = sampling_rate*20, noverlap = sampling_rate*20/2, nfft = None)
    fig, ax = plt.subplots()
    ax.plot(hzPxx, Pxx)
    ax.set_yscale('log')


def compare_pre_post(data_pre, data_post, srate, chan_name):
    nchan_i = channels.index(chan_name)
    x_pre = data_pre[nchan_i,:]
    x_post = data_post[nchan_i,:]
    time = np.arange(x_pre.shape[0]) / srate

    nwind = int(10*srate)
    nfft = nwind
    noverlap = np.round(nwind/2)
    hannw = scipy.signal.windows.hann(nwind)

    hzPxx, Pxx_pre = scipy.signal.welch(
        x_pre, fs = srate, window = hannw, 
        nperseg = nwind, noverlap = noverlap, nfft = nfft
    )

    hzPxx, Pxx_post = scipy.signal.welch(
        x_post, fs = srate, window = hannw, 
        nperseg = nwind, noverlap = noverlap, nfft = nfft
    )

    # One chan pre-post plot
    fig, ax = plt.subplots(figsize = (12, 6))
    ax.plot(time, x_pre, label = 'Pre')
    ax.plot(time, x_post, label = 'Post')
    ax.set_title(chan_name)
    ax.legend()

    # PSD pre-post plot
    fig, ax = plt.subplots(figsize = (12, 6))
    ax.semilogy(hzPxx, Pxx_pre, label = 'Pre')
    ax.semilogy(hzPxx, Pxx_post, label = 'Post')
    ax.set_title(chan_name)
    ax.legend()


In [ ]:
# Export EEG data to MNE-Python format
EEG_data, trig_idxs, trig_type = generate_data_set()

# info object
ch_types = ['eeg'] * len(channels)
info = mne.create_info(ch_names = channels, sfreq = sampling_rate, ch_types = ch_types)
info.set_montage('standard_1020')

# raw object
raw = mne.io.RawArray(EEG_data, info)

# events object
events = np.column_stack((trig_idxs, np.zeros(len(trig_idxs), dtype = int), trig_type)).astype(int)
event_dict = {1: 'Face',
              2: 'Landscape'}
annotations = mne.annotations_from_events(events, sfreq = sampling_rate, event_desc = event_dict)
raw.set_annotations(annotations)

# save to .fif
exportpath = os.path.join(os.getcwd(), "eeg_data_raw.fif")
raw.save(exportpath, overwrite = True)


In [ ]:
# Load data (from .fif file)
importpath = os.path.join(os.getcwd(), "eeg_data_raw.fif")
raw = mne.io.read_raw_fif(importpath, preload = True)

### **2. VISUALIZE DATA**

In [ ]:
# Channel locations
raw.plot_sensors(
    show_names = True, 
    title = "EEG Channel Locations")

In [ ]:
# From trigger events to MNE annotations
events, event_id = mne.events_from_annotations(raw)

print(f"Found {len(events)} events")
print(f"Event types: {len(event_id)}\n")

print("Event mapping:")
for name, code in event_id.items():
    count = (events[:, 2] == code).sum()
    print(f"  {code:>3}  {name:<10} × {count}")

![Trigger_synch.png](Trigger_synch.png)

In [ ]:
fig = mne.viz.plot_events(events, 
                          sfreq = raw.info['sfreq'], 
                          first_samp = raw.first_samp, 
                          event_id = event_id)


fig.set_size_inches(12, 6) 
plt.show()

In [ ]:
# Plot EEG data with annotations (triggers)
raw.plot(
    n_channels = len(raw.ch_names),
    scalings = dict(eeg = 0.25e-6),
    duration = 30,
    start = 0,
    show = True,
    block = True,
    title = "EEG with Annotations"
)

MANUAL DETECTION - BAD CHANNELS / SEGMENTS

We can manually mark: 
- bad channels by clicking on them, 
- bad segments by opening the annotation box labelled 'a', adding a new annotation labelled 'BAD_' and sliding it onto the epochs of interest. 

In [ ]:
# Plot EEG data with annotations (triggers)
raw.plot(
    n_channels = len(raw.ch_names),
    scalings = dict(eeg = 0.25e-6),
    duration = 30,
    start = 0,
    show = True,
    block = True,
    title = "EEG with Annotations"
)

In [ ]:
fig = raw.compute_psd().plot()
fig.set_size_inches(12, 6)
plt.show()

### **3. PRE-PROCESSING**

##### DOWN-SAMPLING, MONTAGE SELECTION & RE-REFERENCING 

In [ ]:
raw.resample(sfreq = 256) # Resample the data to 256 Hz
raw.set_montage("standard_1020") # Apply a template montage 
#ex = raw.set_eeg_reference('average') # Set the reference

In [ ]:
raw = raw.set_eeg_reference(ref_channels = ['A1', 'A2']) # Set the reference
raw = raw.drop_channels(['A1', 'A2']) # Drop reference channels

In [ ]:
#ex.plot(
#    n_channels = len(raw.ch_names),
#    scalings = dict(eeg = 0.25e-6),
#    duration = 30,
#    start = 0,
#    show = True,
#    block = True,
#    title = "EEG with Annotations"
#)

##### FILTERING

In [ ]:
raw.filter(l_freq = 1, h_freq = None)  # High-pass filter at 1 Hz to remove slow drift 

In [ ]:
raw.plot(
    n_channels = len(raw.ch_names),
    scalings = dict(eeg = 0.25e-6),
    duration = 30,
    start = 0,
    show = True,
    block = True,
    title = "EEG with Annotations"
)

In [ ]:
compare_pre_post(data_pre = EEG_data, 
                 data_post = raw.get_data(), 
                 srate = sampling_rate, 
                 chan_name = 'Fp1')


In [ ]:
raw.notch_filter(freqs = [50], notch_widths = 1, phase = 'zero')  # Notch filter for 50 Hz

In [ ]:
raw.plot(
    n_channels = len(raw.ch_names),
    scalings = dict(eeg = 0.25e-6),
    duration = 30,
    start = 0,
    show = True,
    block = True,
    title = "EEG with Annotations"
)

In [ ]:
compare_pre_post(data_pre = EEG_data, 
                 data_post = raw.get_data(), 
                 srate = sampling_rate, 
                 chan_name = 'Fp1')

In [ ]:
fig = raw.compute_psd().plot()
fig.set_size_inches(12, 6)
plt.show()

##### INDEPENDENT COMPONENT ANALYSIS (ICA)

In [ ]:
ica = mne.preprocessing.ICA(
    n_components = .98, 
    random_state = 42,
    method = "fastica")
ica.fit(raw)

ica.plot_sources(raw)
ica.plot_components()

In [ ]:
ica.apply(raw) # exclude bad component

In [ ]:
compare_pre_post(data_pre = EEG_data, 
                 data_post = raw.get_data(), 
                 srate = sampling_rate, 
                 chan_name = 'Fp1')

In [ ]:
raw.plot(
    n_channels = len(raw.ch_names),
    scalings = dict(eeg = 0.25e-6),
    duration = 30,
    start = 0,
    show = True,
    block = True,
    title = "EEG with Annotations"
)

### **4. ANALYSES / SIGNAL PROCESSING**

##### EPOCHING + BASELINE CORRECTION

In [ ]:
# Define epochs parameters
tmin = -0.5  # start of the epoch (before the stim)
tmax = 1.5   # end of the epoch (after the stim)

# Create epochs
epochs = mne.Epochs(raw, 
                    events = events, 
                    event_id = event_id, 
                    tmin = tmin,
                    tmax = tmax, 
                    baseline = (-0.5, 0), # can be set to 'None' 
                    preload = True, 
                    detrend = None, # 1 = linear, 0 = mean detrending
                    reject_by_annotation = True) # reject epochs marked as bad


In [ ]:
total_epochs_created = len(epochs.drop_log) # Number of epochs 

bad_dropped_epochs = [log for log in epochs.drop_log if 'BAD_' in log] # Number of epochs dropped due to bad annotations
bad_dropped_count = len(bad_dropped_epochs)

epochs_kept_count = total_epochs_created - bad_dropped_count 
proportion_dropped_bad = bad_dropped_count / total_epochs_created

print(f"Total epochs created: {total_epochs_created}")
print(f"Number of epochs dropped due to bad annotations: {bad_dropped_count}")
print(f"Proportion of epochs dropped due to bad annotations: {proportion_dropped_bad:.2%}")

In [ ]:
# If needed to reject bad epochs manually
epochs.plot(n_epochs = 10, events = events, event_id = event_id, scalings = dict(eeg = 0.25e-6))

##### ERP ANALYSIS

>>> **Do faces and landscapes elicit different evoked responses?**

In [ ]:
erp_1 = epochs["Face"].average() # create an evoked object by condition
erp_2 = epochs["Landscape"].average()

In [ ]:
fig1 = erp_1.plot_joint(title = "Face") # plot the evoked responses
fig2 = erp_2.plot_joint(title = "Landscape")

fig1.set_size_inches(12, 6)
fig2.set_size_inches(12, 6)

plt.show()

In [ ]:
# Comparisons (contrast Stimulus 1 vs Stimulus 2)
mne.viz.plot_compare_evokeds([erp_1, erp_2], 
                             title = "Comparison of Evoked Responses", 
                             combine = 'gfp') # global field power = population standard deviation across all sensors, for every time point (alt. use "mean")


>>> **Where are these effects on the scalp?**

In [ ]:
erp_diff = mne.combine_evoked([erp_1, erp_2], weights=[1, -1])

time_plot = [-0.5, 0, 0.150, 0.250, 0.3, 0.5]

mask_params = dict(markersize = 15, markerfacecolor = 'y')
erp_diff.plot_topomap(times = time_plot, show = False, mask_params = mask_params)

In [ ]:
times = erp_1.times

channels = erp_1.ch_names
n_channels = len(channels)

n_cols = 4
n_rows = int(np.ceil(n_channels / n_cols))
fig = plt.figure(figsize = (16, 3.2 * n_rows))

outer_gs = fig.add_gridspec(n_rows, n_cols, wspace = 0.1, hspace = 0.4)


eeg_picks = mne.pick_types(erp_1.info, eeg = True)

topo_pos = mne.channels.layout._find_topomap_coords(erp_1.info, picks = eeg_picks)

topo_names = [
    erp_1.info["ch_names"][i]
    for i in eeg_picks
]

topo_pos_dict = {
    name: topo_pos[i]
    for i, name in enumerate(topo_names)
}

all_pos = np.array(list(topo_pos_dict.values()))

# Plot chan
for ch_i, ch in enumerate(channels):

    row = ch_i // n_cols
    col = ch_i % n_cols

    inner_gs = outer_gs[row, col].subgridspec(
        1,
        2,
        width_ratios = [5, 1.5],
        wspace = 0.05
    )

    ax = fig.add_subplot(inner_gs[0, 0]) # ERP
    ax_topo = fig.add_subplot(inner_gs[0, 1])  # topomap

    ax.plot(times, erp_1.data[ch_i] * 1e6, color = "#40b2d8", linewidth = 1.3)
    ax.plot(times, erp_2.data[ch_i] * 1e6, color = "#f26d41", linewidth = 1.3)
    ax.axvline(0, linestyle = "--", color = "black", linewidth = 0.8)
    ax.set_title(ch, fontsize = 11, fontweight = "bold", loc = "left")
    ax.set_xlim(times[0], times[-1])

    # Only labels on outer edges
    if row == n_rows - 1:
        ax.set_xlabel("Time (s)")
    else:
        ax.tick_params(labelbottom=False)

    if col == 0:
        ax.set_ylabel("µV")
    else:
        ax.tick_params(labelleft=False)


    # Plot all electrodes
    ax_topo.scatter(
        all_pos[:, 0],
        all_pos[:, 1],
        s = 15,
        color = "black"
    )

    if ch in topo_pos_dict:

        x, y = topo_pos_dict[ch]

        ax_topo.scatter(
            x,
            y,
            s = 70,
            facecolors = "none",
            edgecolors = "#ac4bd8",
            linewidths = 1
        )


    ax_topo.set_aspect("equal")

    ax_topo.set_xticks([])
    ax_topo.set_yticks([])

    for spine in ax_topo.spines.values():
        spine.set_visible(False)


fig.legend(
    ["Face", "Landscape"],
    loc = "upper center",
    ncol = 2,
    bbox_to_anchor = (0.5, 0.99)
)

plt.show()

##### TIME-FREQUENCY

>>> **Do faces and landscapes also differ in their oscillatory dynamics?**

In [ ]:
decim = 2 
freqs = np.arange(4, 120, 2)  # frequencies of interest
n_cycles = np.linspace(4, 21, freqs.size) # N cycle / freq
tfr_kwargs = dict(
    method = "morlet", 
    freqs = freqs,
    n_cycles = n_cycles,
    decim = decim, # reduce sampling rate of the time-frequency decomposition by the defined decim 
    return_itc = False,
    average = False
)

tfr_face = epochs["Face"].compute_tfr(**tfr_kwargs)
tfr_landscape = epochs["Landscape"].compute_tfr(**tfr_kwargs)

In [ ]:
tfr_face.apply_baseline(mode = "ratio", baseline = (-0.5, 0)) # ratio = power relative to baseline
tfr_landscape.apply_baseline(mode = "ratio", baseline = (-0.5, 0))

In [ ]:
power_face = tfr_face.data.mean(axis = 0) # averaging across epochs
power_landscape = tfr_landscape.data.mean(axis = 0)

power_diff = power_face - power_landscape 

times = tfr_face.times * 1000  # ms
freqs = tfr_face.freqs
channels = tfr_face.ch_names

n_channels = len(channels)

In [ ]:
times_ms = tfr_face.times * 1000

def plot_full_tfr(
    power_face,
    power_landscape,
    power_diff,
    times_ms,
    freqs,
    channels,
    channel
):

    ch_i = channels.index(channel)  # channel index

    face = power_face[ch_i, :, :] 
    landscape = power_landscape[ch_i, :, :]
    diff = power_diff[ch_i, :, :]

    vmin = min(face.min(), landscape.min())
    vmax = max(face.max(), landscape.max())

    diff_max = np.max(np.abs(diff))

    fig, axes = plt.subplots(1, 3, figsize = (17, 5), sharey = True)

#   Plot Face
    im1 = axes[0].pcolormesh(
        times_ms,
        freqs,
        face,
        shading = "auto",
        vmin = vmin,
        vmax = vmax
    )

    axes[0].set_title(f"Face — {channel}", fontsize = 13, fontweight = "bold")
    axes[0].set_xlabel("Time (ms)")
    axes[0].set_ylabel("Frequency (Hz)")

    fig.colorbar(im1, ax = axes[0], label = "Power / baseline")

#   Plot Landscape
    im2 = axes[1].pcolormesh(
        times_ms,
        freqs,
        landscape,
        shading = "auto",
        vmin = vmin,
        vmax = vmax
    )

    axes[1].set_title(f"Landscape — {channel}", fontsize = 13, fontweight = "bold")
    axes[1].set_xlabel("Time (ms)")

    fig.colorbar(im2, ax = axes[1], label = "Power / baseline")

#   Plot difference
    im3 = axes[2].pcolormesh(
        times_ms,
        freqs,
        diff,
        shading = "auto",
        cmap = "RdBu_r",
        vmin = -diff_max,
        vmax = diff_max
    )

    axes[2].set_title("Face − Landscape", fontsize = 13, fontweight = "bold")
    axes[2].set_xlabel("Time (ms)")

    fig.colorbar(im3, ax = axes[2], label = "Power difference")

    for ax in axes:
        ax.axvline(0, linestyle = "--", linewidth = 1)
        ax.set_ylim(freqs[0], freqs[-1])

    fig.suptitle(
        f"Time-frequency activity — {channel}",
        fontsize = 16,
        fontweight = "bold"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
for channel in tfr_face.ch_names:
    plot_full_tfr(
        power_face = power_face,
        power_landscape = power_landscape,
        power_diff = power_diff,
        times_ms = times_ms,
        freqs = freqs,
        channels = tfr_face.ch_names,
        channel = channel
    )